In [16]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [17]:
spark = SparkSession.builder.remote("sc://localhost:15002").getOrCreate()

In [18]:
CLICKHOUSE_URL = "jdbc:clickhouse://clickhouse-server:8123/ecommercial_warehouse"
CLICKHOUSE_BRONZE_TABLE = "bronze_product_snapshot"
CLICKHOUSE_SILVER_TABLE = "silver_category"
CLICKHOUSE_USER = "clickhouse"
CLICKHOUSE_PASSWORD = "123456"

In [19]:
bronze_product_snapshot_df = (
    spark.read.format("jdbc")
    .option("url", CLICKHOUSE_URL)
    .option("driver", "com.clickhouse.jdbc.ClickHouseDriver")
    .option("dbtable", CLICKHOUSE_BRONZE_TABLE)
    .option("user", CLICKHOUSE_USER)
    .option("password", CLICKHOUSE_PASSWORD)
    .load()
)

In [20]:
category_tree_df = bronze_product_snapshot_df.select(
    bronze_product_snapshot_df.url_id,
    bronze_product_snapshot_df.category_tree,
    bronze_product_snapshot_df.observed_at.alias("valid_from")
)

In [21]:
category = category_tree_df.withColumn("category_sk", F.sha2(F.col("category_tree"), 256))

In [22]:
category = category.dropDuplicates(["url_id", "category_sk"])\
    .select(
        category.url_id,
        category.category_sk,
        F.split(category_tree_df.category_tree, " > ").alias("category_tree"),
        category.valid_from
    )

In [23]:
max_level = category.select(F.max(F.size(category.category_tree))).first()[0]

In [24]:
category_lvs = category.select(
    category.url_id,
    category.category_sk,
    *[F.try_element_at(category.category_tree, F.lit(i + 1)).alias(f"category_level_{i+1}") for i in range(max_level)],
    category.valid_from
)

In [25]:
silver_product_snapshot_df = spark.read.format("jdbc")\
    .option("url", CLICKHOUSE_URL)\
    .option("driver", "com.clickhouse.jdbc.ClickHouseDriver")\
    .option("dbtable", CLICKHOUSE_SILVER_TABLE)\
    .option("user", CLICKHOUSE_USER)\
    .option("password", CLICKHOUSE_PASSWORD)\
    .load()

In [26]:
joined_category = category_lvs.alias("category_lvs").join(silver_product_snapshot_df.alias("silver_product_snapshot_df"), 
                  [
                      category_lvs.url_id == silver_product_snapshot_df.url_id,
                      category_lvs.category_sk == silver_product_snapshot_df.category_sk
                ],
                  how="left")\
    .select("category_lvs.*")

In [27]:
joined_category.orderBy("url_id").show(10, truncate=False)

+------+----------------------------------------------------------------+----------------+----------------------+-----------------------+------------------+----------------+-------------------+
|url_id|category_sk                                                     |category_level_1|category_level_2      |category_level_3       |category_level_4  |category_level_5|valid_from         |
+------+----------------------------------------------------------------+----------------+----------------------+-----------------------+------------------+----------------+-------------------+
|38    |34045890b071b127961671c2e81bfe359d1b4893dcb6aa23f0da565dbaae8ee7|thời trang nam  |áo nỉ - áo len nam    |áo len dệt kim         |NULL              |NULL            |2026-11-07 14:12:48|
|49    |34045890b071b127961671c2e81bfe359d1b4893dcb6aa23f0da565dbaae8ee7|thời trang nam  |áo nỉ - áo len nam    |áo len dệt kim         |NULL              |NULL            |2026-11-07 13:20:18|
|50    |34045890b071b127961671

In [28]:
if joined_category.count() > 0:
    joined_category.withColumn("ingested_at", F.current_timestamp())\
    .write.format("jdbc")\
    .option("url", CLICKHOUSE_URL)\
    .option("driver", "com.clickhouse.jdbc.ClickHouseDriver")\
    .option("dbtable", CLICKHOUSE_SILVER_TABLE)\
    .option("user", CLICKHOUSE_USER)\
    .option("password", CLICKHOUSE_PASSWORD)\
    .mode("append")\
    .save()
    print("Data inserted into silver_category table successfully.")

Data inserted into silver_category table successfully.
